# 04. Polynomial Features: Beyond Straight Lines

**Goals.** By the end you can:

- recognize a dataset where logistic regression's straight-line boundary genuinely fails, and fix it
  with **polynomial features**: still a *linear* model, just on transformed inputs;
- check a comparison across **several train/test splits**, not just one, and build models as
  **pipelines**;
- judge honestly, on real data, whether polynomial features help;
- **on your own**, choose the degree for a new dataset.

**Estimated time:** about 75 minutes.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Interactive sliders (optional): if ipywidgets is missing, we show a few fixed pictures instead.
try:
    from ipywidgets import interact, FloatSlider, FloatLogSlider
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
    print("ipywidgets is not installed: the interactive cells will show fixed pictures instead.")

import pandas as pd
from sklearn.datasets import make_circles, make_moons, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

plt.rcParams["figure.figsize"] = (6, 5)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)

In [ ]:
# --- exercise checker: just run this cell -----------------------------------------------
import base64 as _b64, json as _json
_ANS = _json.loads(_b64.b64decode("eyIyLkEiOiA5LCAiT24geW91ciBvd246IEFVQ3MiOiBbMC45MTYyOTYyOTYyOTYyOTYzLCAwLjkxNTgwMjQ2OTEzNTgwMjQsIDAuOTczODg4ODg4ODg4ODg4OCwgMC45NzUyNDY5MTM1ODAyNDY4LCAwLjk3NDMyMDk4NzY1NDMyMV0sICJPbiB5b3VyIG93bjogZGVncmVlIjogM30="))

def _has_str(v):
    return isinstance(v, str) or (isinstance(v, (list, tuple)) and any(_has_str(x) for x in v))

def _norm(v):
    return [_norm(x) for x in v] if isinstance(v, (list, tuple, np.ndarray)) else str(v).strip().lower()

def _close(a, b, tol):
    if _has_str(b):
        return _norm(a) == _norm(b)
    try:
        return np.allclose(np.asarray(a, dtype=float), np.asarray(b, dtype=float), atol=tol, rtol=0)
    except Exception:
        return False

def check(key, value, tol=1e-3):
    # Compare your answer for exercise `key` with the expected one.
    if value is None or value is Ellipsis or (isinstance(value, (list, tuple)) and any(v is None for v in value)):
        print(f"⏳ {key}: not answered yet")
    elif _close(value, _ANS[key], tol):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. You got {value}; check your work and try again")

def check_against(key, value, reference, tol=1e-3):
    # Compare your answer with a reference value computed in the same cell.
    if value is None or (isinstance(value, (list, tuple)) and any(v is None for v in value)):
        print(f"⏳ {key}: not answered yet")
    elif _close(value, reference, tol):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. You got {value}; check your work and try again")

def check_fn(key, fn, tol=1e-3):
    # Call your function on some test inputs and compare with the expected outputs.
    spec = _ANS[key]
    try:
        outs = [fn(*[np.array(a, dtype=float) for a in args]) for args in spec["args"]]
    except Exception as e:
        print(f"❌ {key}: your function raised an error: {type(e).__name__}: {e}")
        return
    if any(o is None for o in outs):
        print(f"⏳ {key}: not answered yet (your function returns None)")
    elif all(_close(o, ex, tol) for o, ex in zip(outs, spec["out"])):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. Your function gives {outs}; check your work and try again")

## 1. A dataset a straight line can't separate

Every decision boundary we've drawn so far has been a straight line, because logistic regression's
boundary is exactly where $w^T x + b = 0$: linear in $x$. What if the true boundary is a **circle**?

In [ ]:
Xc, yc = make_circles(n_samples=300, noise=0.15, factor=0.4, random_state=42)

plt.scatter(Xc[:, 0], Xc[:, 1], c=yc, cmap="coolwarm", edgecolor="k")
plt.title("Two classes separated by a circle, not a line")
plt.xlabel("feature 1"); plt.ylabel("feature 2")
plt.show()

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(Xc, yc, test_size=0.3, random_state=42, stratify=yc)

clf_lin = LogisticRegression().fit(Xtr, ytr)
print("plain logistic regression test accuracy:", clf_lin.score(Xte, yte))

No better than a coin flip (below 50%!), because no straight line can meaningfully separate a ring
from the disk inside it. Let's see the (useless) boundary it found.

In [ ]:
def plot_boundary(predict_fn, X, y, ax=None, title=""):
    # colour the plane by the predicted probability, and draw the 0.5 boundary
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    x1_min, x1_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    x2_min, x2_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 300), np.linspace(x2_min, x2_max, 300))
    grid = np.c_[xx1.ravel(), xx2.ravel()]
    probs = predict_fn(grid).reshape(xx1.shape)
    cs = ax.contourf(xx1, xx2, probs, levels=20, cmap="coolwarm", alpha=0.65)
    ax.contour(xx1, xx2, probs, levels=[0.5], colors="black", linewidths=2.5)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k")
    ax.set_title(title)
    return cs

plot_boundary(lambda g: clf_lin.predict_proba(g)[:, 1], Xc, yc, title="Plain logistic regression: a straight line can't do this")
plt.show()

**Try it yourself, before reading on:** we're only allowed a *linear* model, but what if we engineer
new features first? Is there a transformation of $(x_1, x_2)$ under which "inside vs. outside a circle"
becomes a *linear* condition? (Hint: the equation of a circle is $x_1^2 + x_2^2 = r^2$. What does that
look like if you treat $x_1^2$ and $x_2^2$ as new features?)

**Your answer:** *(write here)*

## 2. Polynomial features: still a linear model, richer inputs

The trick: instead of feeding $(x_1, x_2)$ into logistic regression, feed in
$(x_1, x_2, x_1^2, x_1 x_2, x_2^2, \dots)$: every monomial up to some degree. Logistic regression still
only ever draws a **linear** boundary, but now it's linear *in the new, expanded feature space*, which
corresponds to a **curved** boundary back in the original $(x_1, x_2)$ space. sklearn's
`PolynomialFeatures` builds these new columns for us.

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)
Xtr_p = poly.fit_transform(Xtr)
Xte_p = poly.transform(Xte)

print("original feature count:", Xtr.shape[1])
print("degree-2 polynomial feature count:", Xtr_p.shape[1])
print("new feature names:", poly.get_feature_names_out(["x1", "x2"]))

clf_poly = LogisticRegression(max_iter=5000).fit(Xtr_p, ytr)
print("degree-2 test accuracy:", clf_poly.score(Xte_p, yte))

In [ ]:
plot_boundary(lambda g: clf_poly.predict_proba(poly.transform(g))[:, 1], Xc, yc,
              title="Logistic regression + degree-2 polynomial features")
plt.show()

A genuinely curved boundary, from a model that is, underneath, still just a linear classifier on an
expanded set of features.

### Exercise 2.A

How many features does `PolynomialFeatures(degree=3, include_bias=False)` make from 2 features?
**Write them down by hand first** ($x_1, x_2, x_1^2, \dots$), then store the number in `n_degree3` and
check it with `PolynomialFeatures`.

In [ ]:
# YOUR CODE HERE
n_degree3 = None
check("2.A", n_degree3)

## 3. How much degree do we need?

More degree means a more flexible boundary, but also more parameters to fit (a risk of overfitting)
and more computation. Let's sweep the degree and look at training vs. test accuracy.

In [ ]:
degrees = [1, 2, 3, 4, 6, 8]
train_accs, test_accs, n_feats = [], [], []

for d in degrees:
    p = PolynomialFeatures(degree=d, include_bias=False)
    Xtr_d = p.fit_transform(Xtr)
    Xte_d = p.transform(Xte)
    clf_d = LogisticRegression(max_iter=5000).fit(Xtr_d, ytr)
    train_accs.append(clf_d.score(Xtr_d, ytr))
    test_accs.append(clf_d.score(Xte_d, yte))
    n_feats.append(Xtr_d.shape[1])

for d, tr, te, nf in zip(degrees, train_accs, test_accs, n_feats):
    print(f"degree={d:<3} n_features={nf:<4} train_acc={tr:.3f}  test_acc={te:.3f}")

plt.plot(degrees, train_accs, "o-", label="train accuracy")
plt.plot(degrees, test_accs, "o-", label="test accuracy")
plt.xlabel("polynomial degree"); plt.ylabel("accuracy")
plt.title("Degree vs. accuracy on the circle dataset")
plt.legend()
plt.show()

Degree 1 (no transformation) fails; degree 2 already solves it almost perfectly, because the *true*
boundary really is a degree-2 curve (a circle). Going well beyond the true degree doesn't buy us
anything here and, on noisier or smaller datasets, would start fitting noise instead of signal: the
classic **overfitting** trade-off. (Remember from 03b: sklearn's `LogisticRegression` applies L2
regularization by default, through `C`, which is the standard way to keep a high-degree polynomial
model from overfitting; a smaller `C` means stronger regularization.)

## 4. Can we trust a *single* train/test split?

So far every improvement we've looked at (circles: from below 50% to 100% on the test set) has been huge and
obvious; one split was plenty to see it. On real data the improvement from polynomial features is
often much smaller, sometimes just a couple of percentage points. Before we compare models that
closely, let's check: how much does accuracy change just from **which random split** we happen to use,
with everything else held fixed?

In [ ]:
# Same model, same data, same everything, except the random train/test split.
accs = []
for seed in range(10):
    Xtr_s, Xte_s, ytr_s, yte_s = train_test_split(Xc, yc, test_size=0.3, random_state=seed, stratify=yc)
    clf = LogisticRegression().fit(Xtr_s, ytr_s)
    accs.append(clf.score(Xte_s, yte_s))

for seed, a in enumerate(accs):
    print(f"split {seed}: accuracy = {a:.3f}")
print(f"\nrange across 10 splits: {min(accs):.3f} to {max(accs):.3f}  (spread of {max(accs)-min(accs):.3f})")

The exact same model swings by several percentage points depending purely on *which 90 test points
happened to land in the test set*. If two models differ by less than that swing, a single split can't
reliably tell them apart: you could get a "winner" just from being lucky (or unlucky) with the split.

**The fix we'll use for the rest of this notebook:** instead of one split, try **several** random
splits and look at the whole spread, not one number. (This idea has a proper name and a more efficient
version, **cross-validation**, which you'll likely see in a later course; "try several splits" is the
same idea in its simplest form.)

**A new tool: pipelines.** `make_pipeline(StandardScaler(), LogisticRegression())` chains the steps
into a single model object. When we call `.fit(X_train, y_train)` on it, it fits the scaler on the
training data, transforms it, and fits the logistic regression on the result. When we call `.predict`
or `.predict_proba` on new data, it transforms that data with the **training** means and standard
deviations first. So it does exactly what we did by hand in 02 (scale using the training part only),
automatically, and without any chance of forgetting a step. That matters because we'll refit on many
different splits: the pipeline re-fits the scaler on the training part of **each** split.
(`poly_model` below adds one more step in the middle, `PolynomialFeatures`.)

In [ ]:
def compare_across_splits(build_models, X, y, n_splits=8, test_size=0.3, seed0=0):
    # Fit each model in build_models (a dict: name -> a function that builds a new model) on n_splits
    # different random train/test splits, and collect accuracy, F1 and AUC for each.
    rows = []
    for i in range(n_splits):
        Xtr_i, Xte_i, ytr_i, yte_i = train_test_split(X, y, test_size=test_size, random_state=seed0 + i, stratify=y)
        for name, build in build_models.items():
            model = build()
            model.fit(Xtr_i, ytr_i)
            pred = model.predict(Xte_i)
            proba = model.predict_proba(Xte_i)[:, 1]
            rows.append({"split": i, "model": name,
                         "accuracy": accuracy_score(yte_i, pred),
                         "f1": f1_score(yte_i, pred),
                         "auc": roc_auc_score(yte_i, proba)})
    return pd.DataFrame(rows)


def summarize(results):
    # mean and standard deviation of each measure, per model
    return results.groupby("model")[["accuracy", "f1", "auc"]].agg(["mean", "std"]).round(4)


def linear_model(C=1.0):
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, C=C))

def poly_model(degree=2, C=1.0):
    return make_pipeline(StandardScaler(), PolynomialFeatures(degree=degree, include_bias=False),
                         LogisticRegression(max_iter=5000, C=C))


def plot_splits(results, metric="accuracy", title=""):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for name, g in results.groupby("model"):
        g = g.sort_values("split")
        ax.plot(g["split"], g[metric], "o-", label=name)
    ax.set_xlabel("split #"); ax.set_ylabel(metric)
    ax.set_title(title)
    ax.legend()
    plt.show()

We'll report three measures for every comparison, averaged over the splits: accuracy, **F1** (the
balance of precision and recall, from 01b) and **AUC** (how well the probabilities rank the positives
above the negatives, at every threshold at once). AUC is our main "which model is better" number,
because it doesn't depend on choosing a threshold.

## 5. Real data: Titanic, does polynomial help?

The passengers of the Titanic: `Pclass` (ticket class), `Sex`, `Age`, `SibSp` and `Parch` (family
aboard), `Fare` and `Embarked` (port), predicting `Survived`. A well-known fact about this dataset:
survival wasn't well explained by any *one* of these features alone. "Women and children first"
applied far more strongly in some classes than in others. That's an **interaction** between `Age`,
`Sex` and `Pclass`: exactly the kind of structure polynomial features (which add *products* of
features, not just the features themselves) can capture, and a plain linear model can't.

In [ ]:
titanic = pd.read_csv("../data/titanic.csv")
titanic = titanic.dropna(subset=["Fare"])  # one incomplete row
print(titanic.shape)
titanic.head()

In [ ]:
# A quick look at the "women and children first, unevenly by class" story before we model it:
titanic["is_child"] = titanic["Age"] < 16
child_survival = titanic.groupby(["Pclass", "is_child"])["Survived"].mean().unstack()
child_survival.columns = ["adults", "children"]
print("survival rate by class and (child vs adult):")
print(child_survival.round(2))

Children survived at a much higher rate than adults in every class, but the *size* of that gap is
very different from class to class. That "it depends on the class" pattern is an interaction: not
something a model that is linear in `Age` and `Pclass` can represent on its own.

**A note on categorical features.** `Sex` has two values, so a single 0/1 column (`Sex_male`) is
enough. `Embarked` (the port where the passenger boarded) has **three** values: `C`, `Q` and `S`. We
can't just code them as 1, 2, 3: that would tell the model that `Q` is "between" `C` and `S`, which is
meaningless. Instead we use **one-hot encoding**: one 0/1 column per value (`Embarked_C`, `Embarked_Q`,
`Embarked_S`), with a 1 in the column of the passenger's port. `pd.get_dummies` does exactly this.
(`dummy_na=True` adds one more column for passengers whose port is unknown.)

In [ ]:
y_titanic = titanic["Survived"].values
num_cols = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
X_titanic = titanic[num_cols].copy()
X_titanic["Age"] = X_titanic["Age"].fillna(X_titanic["Age"].median())
X_titanic["Sex_male"] = (titanic["Sex"] == "male").astype(int)
X_titanic = pd.concat([X_titanic, pd.get_dummies(titanic["Embarked"], prefix="Embarked", dummy_na=True)], axis=1)
X_titanic = X_titanic.astype(float).values

print("feature count:", X_titanic.shape[1], " rows:", X_titanic.shape[0])

In [ ]:
results_titanic = compare_across_splits(
    {"linear (baseline)": linear_model, "degree-2 polynomial": lambda: poly_model(C=0.3)},
    X_titanic, y_titanic, n_splits=8)

print(summarize(results_titanic))
plot_splits(results_titanic, "auc", "Titanic: AUC across 8 random splits")

The polynomial model comes out ahead on AUC in most (not all: that's expected, and part of why we
look at several splits) of the 8 runs, and its *average* AUC is a bit higher. The improvement is modest,
about one AUC point, but it's a real, repeatable effect from letting the model represent
`Age`×`Pclass` and `Age`×`Sex` style interactions, not a fluke of one lucky split.

## 6. When it doesn't help: breast cancer

Polynomial features are not magic. On the Breast Cancer data from 02, they **don't** help:

- With all 30 features, degree 2 makes about 500 features. With a small training set that's more
  parameters than examples, and the polynomial model does **worse** than the linear one (overfitting).
- Even done fairly (only the 2 most informative features, plenty of data), linear and degree-2 come
  out **tied**: the relationship between these tumor measurements and malignancy is already close to
  linear, so there's no curved structure for polynomial features to capture.

Polynomial features pay off only when **both** hold: there's genuine non-linear structure in the
data, *and* there's enough data (or strong enough regularization) for the extra parameters. (The full
version of this notebook runs these experiments.)

## On your own: the two moons

The cell below makes a new 2D dataset: two interleaving half-moons, with some noise.

*You have:* `compare_across_splits`, `summarize` and `poly_model` from section 4.

1. Plot the data. Before computing anything: which degree do you expect to need?
2. For each degree in `degrees_moons`, compare `poly_model(degree=d)` over **8 splits** with
   `compare_across_splits`, and store the **mean AUC** of each degree, in order, in the list
   `mean_auc_moons`.
3. Choose the **smallest** degree whose mean AUC is within 0.005 of the best one (simpler is better when
   the difference is within the noise), and store it in `best_degree`.
4. Draw the boundary of a model with that degree, trained on all the data (`plot_boundary`).

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.3, random_state=0)
degrees_moons = [1, 2, 3, 4, 5]

In [ ]:
# YOUR CODE HERE
mean_auc_moons = None
best_degree = None
print("chosen degree:", best_degree)
check("On your own: AUCs", mean_auc_moons, tol=0.002)
check("On your own: degree", best_degree)

**Your answer:** *(write here)*

## Going further

The **full version** of this notebook (`04_polynomial_features_full_student.ipynb`) also has these parts, for anyone who wants more:

- A 30-second recap of the evaluation measures (01b)
- Ionosphere: a much bigger effect
- Breast cancer: when it doesn't help (the experiments)

## Recap

- Polynomial features let a linear model (logistic regression) fit curved boundaries, by giving it
  $x_1^2, x_1 x_2, x_2^2, \dots$ as additional inputs. The boundary is still linear in the **new**
  features.
- **A single train/test split isn't enough** to trust a comparison, especially when the effect is
  small: use several random splits, and look at the spread, not one number.
- **Pipelines** (`make_pipeline`) chain scaling, polynomial features and the model, so every split is
  handled correctly.
- Compare models with **AUC** (and F1), not accuracy alone.
- **Titanic**: a modest but real, repeatable improvement, from interactions such as `Age`×`Pclass`.
- Polynomial features pay off only when **both** conditions hold: there's genuine non-linear structure
  in the data, *and* there's enough data (or strong enough regularization) for the extra parameters.
- **Next:** the feature-engineering unit, where choosing the right features (the **representation**)
  becomes the main topic.

## References and further reading

- MLU-Explain, [The Bias-Variance Tradeoff](https://mlu-explain.github.io/bias-variance/): underfitting
  and overfitting, visually.
- scikit-learn:
  [`PolynomialFeatures`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)
  and [`make_pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html).